### SFOS API - Network Services Management

**Purpose:** Create, read, update, and delete network services

**Topics:**
- Basic CRUD operations using domain StrEnum objects
- Service creation with various protocols
- Pagination and filtering
- Error handling
- Bulk operations

**Requirements:**
- Environment variables configured (.env file)
- SFOS firewall accessible
- API key with appropriate permissions

---

#### Setup: Import Libraries and Configure Client

In [ ]:

from common import print_data, setup_client
from sfos_api import Network

# Initialize the SFOS API Client
client = setup_client()
print("✅ SFOS API client initialized")

#### Create Network Services

Create service objects using the `Network.SERVICES` endpoint object.

Services define protocols and ports used in firewall rules.

API Link: [Create a Service | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/create-service/)

In [ ]:
# Create a TCP service
endpoint = Network.SERVICES
service_name = "test-service-http-custom"

payload = {
    "name": service_name,
    "description": "Custom HTTP service on port 8080",
    "type":"tcpOrUdp",

    "services":[{"protocol": "tcp",
    "destinationPort": {"to":8080}},]
}

try:
    response = client.create(endpoint, payload)
    print(f"✅ Created service '{service_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating service: {e}")
    print_data(str(e))

In [ ]:
# Create a UDP service
endpoint = Network.SERVICES
service_name = "test-service-dns-custom"

payload = {
    "name": service_name,
    "description": "Custom DNS service",
     "type": "tcpOrUdp",
     "services":[{
    "protocol": "udp",
    "destinationPort":{ "to":5353}}],
}

try:
    response = client.create(endpoint, payload)
    print(f"✅ Created service '{service_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating service: {e}")
    print_data(str(e))

In [ ]:
# Create a service with port range
endpoint = Network.SERVICES
service_name = "test-service-range"

payload = {
    "name": service_name,
    "description": "Service with port range",
      "type": "tcpOrUdp",
    'services' :[{"protocol" :"tcp",
    "destinationPort": {"from":5000,"to":6000}}],
}

try:
    response = client.create(endpoint, payload)
    print(f"✅ Created service '{service_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating service: {e}")
    print_data(str(e))

#### Read Network Services

Retrieve service information by ID/name or list all services.

API Links: [Get all Services | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/get-all-services/) | [Get a Service | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/get-service-by-id-or-name/)

In [ ]:
# Get a specific service by name
endpoint = Network.SERVICES
service_name = "test-service-http-custom"

try:
    response = client.read(endpoint, id_or_name=service_name)
    print(f"✅ Retrieved service '{service_name}'")
    print_data(response)
except Exception as e:
    print(f"Error reading service: {e}")
    print_data(str(e))

In [ ]:
# Get all services
endpoint = Network.SERVICES

try:
    response = client.read(endpoint)

    if "items" in response and response["items"]:
        print(f"✅ Retrieved {len(response['items'])} services")
        # Show first 5 services
        print_data(response["items"][:5],keys=["name","type","services"])
        if len(response["items"]) > 5:
            print(f"... (showing first 5 of {len(response['items'])} services)")
    else:
        print("No services found")
except Exception as e:
    print(f"Error reading services: {e}")
    print_data(str(e))

In [ ]:
# Get services with filtering
endpoint = Network.SERVICES
params = {
    "nameContains": "test-service-",
    "page": 1,
    "pageSize": 10,
}

try:
    response = client.read(endpoint, params=params, pageTotal=False)

    if "items" in response:
        print(f"✅ Retrieved {len(response['items'])} services containing 'test-service-'")
        print_data(response["items"],keys=["name","type","services"])
    else:
        print("No services found matching filter")
except Exception as e:
    print(f"Error reading services: {e}")
    print_data(str(e))

#### Update Network Service

Modify an existing service's properties.

API Link: [Update a Service | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/update-service-by-id-or-name/)

In [ ]:
# Update a service
endpoint = Network.SERVICES
service_name = "test-service-http-custom"

payload = {
    "description": "Updated HTTP service - now on port 9090",

   "services":[{"protocol":"tcp", "destinationPort":{ "to":9090}}],
}

try:
    response = client.update(endpoint, id_or_name=service_name, data=payload)
    print(f"✅ Updated service '{service_name}'")
    print_data(response)
except Exception as e:
    print(f"Error updating service: {e}")
    print_data(str(e))

#### Delete Network Services

Remove service objects (single or bulk delete).

API Links: [Delete a Service | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/delete-service-by-id-or-name/) | [Bulk delete Services | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/bulk-delete-services/)

In [ ]:
# Delete a single service
endpoint = Network.SERVICES
service_name = "test-service-http-custom"

try:
    response = client.delete(endpoint, id_or_name=service_name)
    print(f"✅ Deleted service '{service_name}'")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error deleting service: {e}")
    print_data(str(e))

In [ ]:
# Bulk delete services
endpoint = Network.SERVICES

items_to_delete = [
    {"name": "test-service-http-custom"},
    {"name": "test-service-dns-custom"},
    {"name": "test-service-range"},
]

try:
    response = client.delete(endpoint, items=items_to_delete)
    print(f"✅ Bulk deleted {len(items_to_delete)} services")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error bulk deleting services: {e}")
    print_data(str(e))

In [ ]:
# Close the client session when done
client.close()
print("✅ Client session closed")